In [1]:
library(BiocManager)
library(regioneR)
library(GenomicRanges)
library(rtracklayer)
library(dplyr)
library(tidyr)
library(ggplot2)

Bioconductor version '3.18' is out-of-date; the current release version '3.23'
  is available with R version '4.6'; see https://bioconductor.org/install

Loading required package: GenomicRanges

Loading required package: stats4

Loading required package: BiocGenerics

Loading required package: generics


Attaching package: ‘generics’


The following objects are masked from ‘package:base’:

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union



Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, is.unsorted, lapply, Map, mapply, match, mget,
    order, paste, pmax, pmax.int, pmin, pmin.int, Position, rank,
    rbind, Reduce, rownames, sapply, saveRDS, sort, table, tap

In [32]:
linkedpeaks=read.csv('/nfs/team292/projects/PanTissue/data/temp/ATAC/processed/nonfetal_all/ArchRproj_subset/genefiles/peak2gene_links.csv')
peaks=read.csv('/nfs/team292/projects/PanTissue/data/temp/ATAC/processed/nonfetal_all/ArchRproj_subset/genefiles/peaks.csv')

In [33]:
linkedpeaks=linkedpeaks %>%
	#filter(Correlation>0.75) %>%
	filter(FDR<1e-5) %>%
	distinct(peakName,geneName) %>%
    separate(peakName, into = c("chr", "start", "end"), sep = "[:-]") %>%
    mutate(start = as.integer(start), end = as.integer(end)) %>%
    select(chr, start, end,geneName)

In [34]:
peaks=peaks %>%
    separate(peak, into = c("chr", "start", "end"), sep = "[:-]") %>%
    mutate(start = as.integer(start), end = as.integer(end)) %>%
    select(chr, start, end)

In [7]:
crispr1=read.table("/nfs/team292/projects/PanTissue/data/freeze/ATAC/resources/EPCrisprBenchmark_combined_data.heldout_5_cell_types.GRCh38.tsv",sep='\t',header=T)
crispr2=read.table("/nfs/team292/projects/PanTissue/data/freeze/ATAC/resources/EPCrisprBenchmark_combined_data.training_K562.GRCh38.tsv",sep='\t',header=T)

In [29]:
crispr=rbind(crispr1,crispr2) %>% select(chrom,chromStart,chromEnd,measuredGeneSymbol,Significant,CellType) 

In [ ]:
crispr

In [37]:
head(crispr)

,chrom,chromStart,chromEnd,measuredGeneSymbol,Significant,CellType
,<chr>,<int>,<int>,<chr>,<lgl>,<chr>
1,chr10,78953049,78953549,PPIF,FALSE,GM12878
2,chr10,78973158,78973947,PPIF,FALSE,GM12878
3,chr10,78974475,78974975,PPIF,FALSE,GM12878
4,chr10,78981166,78981817,PPIF,FALSE,GM12878
5,chr10,78991120,78991620,PPIF,FALSE,GM12878
6,chr10,78992272,78992772,PPIF,FALSE,GM12878


In [21]:
head(linkedpeaks)

,chr,start,end,geneName
,<chr>,<int>,<int>,<chr>
1,chr1,905195,905695,FAM87B
2,chr1,980609,981109,FAM87B
3,chr1,1014969,1015469,FAM87B
4,chr1,856520,857020,LINC01128
5,chr1,1142448,1142948,LINC01128
6,chr1,881890,882390,AL645608.6


In [36]:
library(GenomicRanges)
library(dplyr)

# 1. Convert to GRanges
gr_crispr <- GRanges(
  seqnames = crispr$chrom,
  ranges   = IRanges(start = crispr$chromStart, end = crispr$chromEnd),
  gene        = crispr$measuredGeneSymbol,
  Significant = crispr$Significant,
  CellType    = crispr$CellType
)

gr_linked <- GRanges(
  seqnames = linkedpeaks$chr,
  ranges   = IRanges(start = linkedpeaks$start, end = linkedpeaks$end),
  gene     = linkedpeaks$geneName
)

gr_peaks <- GRanges(
  seqnames = peaks$chr,
  ranges   = IRanges(start = peaks$start, end = peaks$end)
)

# 2. Subset CRISPR to peaks that overlap your peaks universe
hits_cp <- findOverlaps(gr_crispr, gr_peaks)
gr_crispr_sub <- gr_crispr[unique(queryHits(hits_cp))]

# 3. For each crispr peak in the subset, check if an overlapping peak
#    in linkedpeaks shares the same gene
hits_cl <- findOverlaps(gr_crispr_sub, gr_linked)

same_gene     <- gr_crispr_sub$gene[queryHits(hits_cl)] ==
                 gr_linked$gene[subjectHits(hits_cl)]
in_linked_idx <- unique(queryHits(hits_cl)[same_gene])

crispr_df <- as.data.frame(gr_crispr_sub)
crispr_df$in_linkedpeaks <- seq_len(nrow(crispr_df)) %in% in_linked_idx

# 4. Fisher's exact test
contingency <- table(
  Significant   = crispr_df$Significant,
  InLinkedPeaks = crispr_df$in_linkedpeaks
)

print(contingency)
print(fisher.test(contingency))

           InLinkedPeaks
Significant FALSE  TRUE
      FALSE 11487   290
      TRUE    535   105

	Fisher's Exact Test for Count Data

data:  contingency
p-value < 2.2e-16
alternative hypothesis: true odds ratio is not equal to 1
95 percent confidence interval:
 6.054492 9.919625
sample estimates:
odds ratio 
  7.772033 



In [ ]:
library(GenomicRanges)
library(dplyr)
library(ggplot2)

# 1. Convert peaks and linked peaks to GRanges (same for all cell types)
gr_linked <- GRanges(
  seqnames = linkedpeaks$chr,
  ranges   = IRanges(start = linkedpeaks$start, end = linkedpeaks$end),
  gene     = linkedpeaks$geneName
)

gr_peaks <- GRanges(
  seqnames = peaks$chr,
  ranges   = IRanges(start = peaks$start, end = peaks$end)
)

# 2. Loop over cell types
celltypes <- unique(crispr$CellType)

results <- lapply(celltypes, function(ct) {
  
  # Subset crispr to this cell type
  crispr_ct <- crispr[crispr$CellType == ct, ]
  
  gr_crispr_ct <- GRanges(
    seqnames = crispr_ct$chrom,
    ranges   = IRanges(start = crispr_ct$chromStart, end = crispr_ct$chromEnd),
    gene        = crispr_ct$measuredGeneSymbol,
    Significant = crispr_ct$Significant
  )
  
  # Subset CRISPR to peaks overlapping your peaks universe
  hits_cp <- findOverlaps(gr_crispr_ct, gr_peaks)
  if (length(hits_cp) == 0) return(NULL)
  gr_crispr_sub <- gr_crispr_ct[unique(queryHits(hits_cp))]
  
  # Check which crispr pairs overlap a linkedpeaks pair with the same gene
  hits_cl <- findOverlaps(gr_crispr_sub, gr_linked)
  if (length(hits_cl) == 0) return(NULL)
  
  same_gene     <- gr_crispr_sub$gene[queryHits(hits_cl)] ==
                   gr_linked$gene[subjectHits(hits_cl)]
  in_linked_idx <- unique(queryHits(hits_cl)[same_gene])
  
  crispr_df <- as.data.frame(gr_crispr_sub)
  crispr_df$in_linkedpeaks <- seq_len(nrow(crispr_df)) %in% in_linked_idx
  
  # Fisher's exact test
  contingency <- table(
    Significant   = crispr_df$Significant,
    InLinkedPeaks = crispr_df$in_linkedpeaks
  )
  
  # Guard: need all four cells populated
  if (!all(c("FALSE", "TRUE") %in% rownames(contingency)) ||
      !all(c("FALSE", "TRUE") %in% colnames(contingency))) return(NULL)
  
  ft <- fisher.test(contingency)
  
  data.frame(
    CellType = ct,
    OR       = ft$estimate,
    CI_low   = ft$conf.int[1],
    CI_high  = ft$conf.int[2],
    pval     = ft$p.value,
    n_linked = sum(crispr_df$in_linkedpeaks),
    n_total  = nrow(crispr_df)
  )
})


results_df <- bind_rows(results) %>%
  mutate(
    sig_label = ifelse(pval < 0.05, "*", ""),
    CellType  = reorder(CellType, OR)
  )

print(results_df)

celltype_labels <- c(
  GM12878 = "GM12878 (Immune lymphoid)",
  HCT116  = "HCT116 (Epithelial colorectal)",
  Jurkat  = "Jurkat (Immune lymphoid)",
  K562    = "K562 (Immune myeloid)",
  WTC11   = "WTC11 (Pluripotent iPSC)"
)
results_df <- results_df %>%
  mutate(
    CellTypeLabel = celltype_labels[as.character(CellType)],
    CellTypeLabel = reorder(CellTypeLabel, OR)
  )

               CellType        OR    CI_low   CI_high         pval n_linked
odds ratio...1  GM12878  3.423731 0.6433178 19.735821 1.163423e-01        9
odds ratio...2   HCT116  6.331678 1.5462769 23.047363 5.164650e-03       14
odds ratio...3   Jurkat  3.759910 0.2976209 30.456754 1.752444e-01        8
odds ratio...4     K562  7.374161 5.5848586  9.666158 3.467708e-37      318
odds ratio...5    WTC11 15.080548 5.4764101 37.817341 6.774146e-07       46
               n_total sig_label
odds ratio...1      66          
odds ratio...2     327         *
odds ratio...3      71          
odds ratio...4   10227         *
odds ratio...5    1726         *


In [55]:
p=ggplot(results_df, aes(x = OR, y = CellTypeLabel)) +
  geom_vline(xintercept = 1, linetype = "dashed", colour = "grey50") +
  geom_errorbarh(aes(xmin = CI_low, xmax = CI_high),
                 height = 0.1, colour = "grey40", linewidth = 0.4) +
  geom_point(aes(size = n_linked), shape = 18, colour = "#3a86ff") +
  geom_text(aes(label = sig_label, x = CI_high),
            hjust = -0.4, size = 5, show.legend = FALSE) +
  scale_x_log10() +
  labs(
    x    = "Odds ratio (validated peak-gene links)",
    y    = "Cell line",
    size = "N linked pairs"
  ) +
  theme_classic() +
  theme(
    axis.text.y  = element_text(size = 10),
    axis.text.x  = element_text(size = 10),
    axis.title   = element_text(size = 11)
  )

In [56]:
ggsave('/nfs/team292/projects/PanTissue/results/freeze/ATAC/plots/CRISPR_validated_vs_unvalidated_OR.pdf',p,width=5.2,height=2)

In [ ]:
log2(10)

[1] 2.302585